# Fake News Detection - Colab runner

Code comes from GitHub (re-running the setup cell pulls the latest commit).
The big dataset lives in Google Drive, and outputs are copied back to Drive.

**Before running:** `Runtime > Change runtime type > T4 GPU`.

**One-time:** put `WELFake_Dataset.csv` in Drive at `MyDrive/fake-news-data/WELFake_Dataset.csv`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL = 'https://github.com/mehediparvez/fake_news_detection.git'
BRANCH = 'main'
PROJECT_DIR = '/content/fake_news_detection'
DRIVE_DATA = '/content/drive/MyDrive/fake-news-data/WELFake_Dataset.csv'
DRIVE_OUT = '/content/drive/MyDrive/fake-news-data/outputs'

# Clone the first time, otherwise fetch and hard-reset to the latest GitHub commit.
if not os.path.isdir(PROJECT_DIR + '/.git'):
    !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin
    !git -C {PROJECT_DIR} reset --hard origin/{BRANCH}

os.chdir(PROJECT_DIR)
os.makedirs('data', exist_ok=True)
!cp -n "{DRIVE_DATA}" data/WELFake_Dataset.csv
!git log -1 --oneline
!ls -la data

In [ ]:
# Colab already has torch, pandas, scikit-learn, seaborn and joblib.
!pip install -q -r requirements.txt
!python check_setup.py

## 1. Clean the dataset

In [ ]:
!python prepare_dataset.py

## 2. Train the TF-IDF + Logistic Regression baseline

In [ ]:
!python train_baseline.py

## 3. Train the BiLSTM (should print `Device: cuda`)

In [ ]:
!python train_bilstm.py

## 4. Charts

In [ ]:
!python visualize_results.py

from IPython.display import Image, display
for name in ['accuracy_comparison.png', 'model_metrics_comparison.png', 'bilstm_confusion_matrix.png']:
    path = os.path.join(PROJECT_DIR, 'reports', name)
    if os.path.exists(path):
        display(Image(path, width=600))

## 5. Save models and reports to Drive

Colab's disk is wiped when the session ends.

In [ ]:
!mkdir -p "{DRIVE_OUT}"
!cp -r models reports "{DRIVE_OUT}"/
!ls "{DRIVE_OUT}"

## 5b. Package the full project as one zip in Drive

Code + trained models + reports + dataset, so anyone can download it and run it locally (`pip install -r requirements.txt`, then `streamlit run app.py`).

In [ ]:
!cd /content && zip -rq "{DRIVE_OUT}/fake-news-detection-full.zip" fake_news_detection -x "fake_news_detection/.git/*"
!ls -lh "{DRIVE_OUT}/fake-news-detection-full.zip"

## 6. Launch the Gradio app

Prints a public `https://xxxx.gradio.live` link (valid ~1 week while this cell keeps running). Stop the cell to shut it down.

In [ ]:
!python gradio_app.py --share